# 🏛️ **Chronos AI Project - Part III: Volumetric Reconstruction and Metrology**

In the previous stages of this project (Parts I and II), the focus of the Artificial Intelligence was on **Macro-Prospection**. We used clustering algorithms (*DBSCAN*) to scan vast expanses of virtual terrain, filtering out geological noise to isolate "anomalies".

However, for Archaeology and Engineering, a Point Cloud is still an abstraction. The field professional needs to calculate the volume of earth to be removed, dimension the structure, and visualize its continuous geometry. In this **Part III**, we elevate Chronos to an **Analytical Tool for Micro-Excavation and Automated Topography**.

---


## ☝ **Phase 1: The Geometric Prototype (Alpha Shapes and Convex Hull)**

Before we deal with "Spatial Big Data" from industrial radars, we need to understand the mathematics of how the computer transforms floating points ($R^3$) into a solid and tangible surface (a triangle mesh).

For this proof of concept, we will use **Alpha Shapes** (Concave Hulls) and **Delaunay Triangulation**.
* Unlike the *Convex Hull* (which wraps the points like a stretched box), the *Alpha Shape* allows an imaginary "sphere" of adjustable radius to roll through the point cloud, revealing valleys, curves, and concavities of the artifacts.

In [1]:
# pip install alphashape

### 🛠️ **The Mathematical and Visual Ecosystem**

To materialize the geometric abstraction of the prototype, we need an ecosystem of libraries that acts on two simultaneous fronts: structural calculation and interactive rendering.

In this first step of Phase 1, our technology stack is divided into three fundamental pillars:

1. The Algebraic Foundation (`numpy` and `pandas`): Every point cloud is, in essence, a matrix of coordinates in the $R^3$ space.

    - **NumPy** acts as our linear algebra engine, processing these vectors in fractions of a second to ensure the pipeline's performance.

2. Topological Intelligence (`alphashape` and `scipy.spatial`): Here lies the core of the reconstruction. The **AlphaShape** package calculates the concave hull around the data.

    - Under the hood, it resorts to **Delaunay Triangulation** (via *SciPy*) to optimally connect the vertices, avoiding geometric distortions that would ruin the volumetry of the ruin.

3. Geometric Rendering (`plotly.graph_objects`): Pure mathematics needs visual validation. **Plotly** works as our interactive 3D laboratory.

    - It allows rotating the artifacts, applying opacity, and superimposing the generated mesh over the original raw radar points for immediate visual auditing.

In [2]:
# ============================================================================
# CHRONOS — SETUP
# ============================================================================
import sys, random, warnings

# --- Environment check ---------------------------------------------------------
_PY = sys.version_info
if _PY >= (3, 13):
    warnings.warn(
        "\n" + "=" * 74 +
        "\n  Python 3.13 detected. Open3D 0.19 has no wheel for this version,"
        "\n  so Part III will not run. Use Python 3.10-3.12:"
        "\n  python3.12 -m venv .venv && source .venv/bin/activate"
        "\n" + "=" * 74,
        stacklevel=2)

# --- Numerics ---------------------------------------------------------
import numpy as np
import pandas as pd

# --- Plotting ---------------------------------------------------------
import matplotlib
import matplotlib.pyplot as plt

# matplotlib registers the '3d' projection by importing mpl_toolkits.mplot3d.
# Importing it explicitly turns a confusing ValueError into a clear one.
try:
    from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
    _HAS_3D = "3d" in matplotlib.projections.get_projection_names()
except ImportError:
    _HAS_3D = False

if not _HAS_3D:
    raise ImportError(
        "\n" + "=" * 74 +
        "\n  matplotlib cannot register the '3d' projection."
        "\n  Cause: mpl_toolkits.mplot3d is missing or shadowed - usually two"
        "\n  matplotlib installs (system + pip --user, as in ~/.local)."
        "\n"
        "\n  Fix:"
        "\n      pip uninstall -y matplotlib"
        "\n      pip install 'matplotlib>=3.7,<4.0'"
        "\n" + "=" * 74)

plt.style.use("ggplot")

import plotly.io as pio
import plotly.graph_objects as go
import alphashape
from scipy.spatial import ConvexHull, Delaunay
from sklearn.cluster import DBSCAN

try:
    import google.colab                              # noqa: F401
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "notebook_connected"

# --- Open3D -----------------------------------------------------------
# Open3D 0.19 ships wheels for cp38-cp312 only. It cannot install on 3.13+.
try:
    import open3d as o3d
    HAS_OPEN3D = True
except ImportError:
    o3d = None
    HAS_OPEN3D = False
    print("\n" + "=" * 74)
    print("  Open3D is not available. The 3D reconstruction cells will be skipped.")
    print("  Open3D 0.19 supports Python 3.8-3.12 only (no cp313 wheel).")
    print("  Fix: python3.12 -m venv .venv && pip install -r requirements.txt")
    print("=" * 74 + "\n")

# --- Reproducibility ---------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Python {_PY.major}.{_PY.minor}.{_PY.micro} | matplotlib {matplotlib.__version__}")
print(f"Seed {SEED} | renderer: {pio.renderers.default}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Python 3.11.15 | matplotlib 3.10.8
Seed 42 | renderer: notebook_connected


### 🌘 **The Challenge of Concavities: The Topology of Alpha Shapes**

### Surface Tension in Reconstruction

> *In archaeology, the true identity of an artifact lies in its imperfections, recesses, and hidden valleys.*

When trying to digitally "envelope" real discoveries — such as the curved fragment of a ceramic amphora or the collapsed vault of a ruin — we face a severe topological problem. Archaeological artifacts are almost never perfectly convex polyhedrons. If we apply a rigid approach (like wrapping the object in tightly stretched plastic wrap), the algorithm will create a closed block that "swallows" the natural curves and ignores the internal geometry of the discovery.

To overcome this obstacle, we enter the domain of Advanced Computational Geometry with the concept of **Alpha Shapes**.

### 📐 **The Mathematics of Shape** (The $\alpha$ Parameter)

The scalar parameter $\alpha$ (Alpha) works as a "tension" control for our 3D mesh. Mathematically, it defines the inverse radius ($R = 1/\alpha$) of a theoretical sphere that rolls over the external surface of the point cloud, sculpting the empty space.

In the algorithmic laboratory below, we forge a stochastic point cloud in the shape of a "half-moon" (simulating a noisy ceramic fragment) to visually validate this interaction between mathematics and shape.

### **The Control Variable** (`alpha_value`):

The calibration of this parameter is critical for the fidelity of the reconstruction:

**Absolute Convexity ($\alpha = 0.0$):**
*   The mesh tension is infinite (Radius $R \to \infty$).
*   The algorithm ignores any concavity and encloses the "moon" in a strict polygon without recesses (Convex Hull).
*   **Result:** Total loss of morphological details.

**Topological Adjustment ($\alpha \approx 0.5$):**
*   The sphere's radius decreases, allowing the mesh to "dive" into the data depressions.
*   Accurately maps the true curved topology of the object, respecting the "emptiness" of the half-moon.
*   **Result:** Faithful reconstruction of the original artifact.

**Structural Degradation ($\alpha > 5.0$):**
*   The radius becomes smaller than the average distance between the points (sensor resolution).
*   The mesh starts to penetrate the data and tear the model, generating unwanted fragmentation and topological holes.
*   **Result:** The object digitally disintegrates.

In [3]:
# --- 1. GENERATE A CURVED SHAPE (BANANA / MOON) ---
# Let's create points forming a curve (Sinusoid)
t = np.linspace(0, np.pi, 200) # Half circle
x = 10 * np.cos(t) + np.random.normal(0, 0.5, 200)
y = 20 * np.sin(t) + np.random.normal(0, 0.5, 200) # Stretch on Y to curve
z = np.random.normal(0, 1, 200) # Thickness in Z

points_3d = np.column_stack((x, y, z))


# --- 2. THE ALPHA TEST ---
# Change this value to see the magic happen:
# alpha = 0.0 -> Will create a closed shape (looks like a 'D')
# alpha = 0.2 -> Will start to understand the curve
# alpha = 5.0 -> Will tear everything apart (holes)
alpha_value = 0.5

print(f"Calculating Alpha Shape with value: {alpha_value}")
try:
    alpha_shape = alphashape.alphashape(points_3d, alpha_value)

    # Extract mesh
    vertices = np.array(alpha_shape.vertices)
    faces = np.array(alpha_shape.faces)

    # Plot
    fig = go.Figure()

    # Mesh
    fig.add_trace(go.Mesh3d(
        x = vertices[:, 0],
        y = vertices[:, 1],
        z = vertices[:, 2],
        i = faces[:, 0],
        j = faces[:, 1],
        k = faces[:, 2],
        color = 'cyan',
        opacity = 0.5,
        name = 'Mesh'
    ))

    # Original Points
    fig.add_trace(go.Scatter3d(
        x = points_3d[:, 0],
        y = points_3d[:, 1],
        z = points_3d[:, 2],
        mode = 'markers',
        marker = dict(size = 3, color = 'red'),
        name = 'Points'
    ))

    fig.update_layout(title = f"Alpha Value: {alpha_value}")
    fig.show()

except Exception as e:
    print(f"Alpha failed (probably too high for the point density): {e}")

Calculating Alpha Shape with value: 0.5


### 🧊 **Volumetric Reconstruction of Underground Structures**

In this step, we apply the geometric foundation of *Alpha Shapes* to a simulated archaeological prospection scenario. The algorithm initializes by forging a cluster of coordinates that mimics the stochastic signature of a radar (GPR), representing a rectangular chamber (tomb) buried at a depth of 4.5 meters.

The main objective of this block is not merely filtering, but the transition from a **discrete representation** (point cloud) to a **continuous representation** (polygonal mesh or *Mesh*).

For modern graphics engines to process the three-dimensional reconstruction, the topology calculated by the algorithm must be decomposed into two fundamental matrix structures:

1.  **Vertex Matrix ($x, y, z$):** Mapping of the real spatial coordinates in the Euclidean space.
2.  **Face Matrix ($i, j, k$):** Index structure based on Delaunay Triangulation, which acts as the connectivity rule. It instructs the renderer on which trios of vertices must be connected to close a solid plane (face).

> 💡 **Engineering Note:** During rendering in *Plotly*, the spatial integrity of the structure is ensured by the `aspectmode='data'` configuration. This parameter forces the isometry of the axes (1:1:1 scale), a non-negotiable methodological requirement in engineering to ensure that the model does not suffer visual distortions that would invalidate the structural analysis of the discovery.

In [4]:
# --- 1. GENERATING TOMB DATA ---
points_3d = []

# Tomb: A slightly deformed rectangular box
for _ in range(200):
    x = 40 + np.random.normal(0, 0.5)
    y = 40 + np.random.normal(0, 0.5)
    z = -4.5 + np.random.normal(0, 0.5)
    points_3d.append([x, y, z])

points = np.array(points_3d)

print(f"📡 Generated Points: {len(points)}. Starting mesh reconstruction...")


# --- 2. THE ALPHA SHAPE ALGORITHM ---
# alpha = 0.0 -> Convex Hull (wraps everything in tight wrapping paper)
# alpha > 0.0 -> Concave Hull (starts detailing the curves)
alpha_value = 0.8

# This magic function generates the 3D object
alpha_shape = alphashape.alphashape(points, alpha_value)


# --- 3. EXTRACTING VERTICES AND FACES FOR PLOTLY ---
# Plotly needs to know: Where the points are (Vertices) and what connects to what (Faces/Triangles)
vertices = np.array(alpha_shape.vertices)
faces = np.array(alpha_shape.faces)

x, y, z = vertices[:, 0], vertices[:, 1], vertices[:, 2]
i, j, k = faces[:, 0], faces[:, 1], faces[:, 2]     # Triangle indices

print(f"📐 Mesh generated with {len(faces)} triangular faces.")


# --- 4. REAL VISUALIZATION (SOLID) ---
fig = go.Figure()

# Adds the MESH
fig.add_trace(go.Mesh3d(
    x = x,
    y = y,
    z = z,
    i = i,
    j = j,
    k = k,
    color = 'gold',
    opacity = 0.50,
    name = 'Reconstructed Structure',
    showscale = True
))

# Adds the original POINTS (for comparison)
fig.add_trace(go.Scatter3d(
    x = points[:, 0],
    y = points[:, 1],
    z = points[:, 2],
    mode = 'markers',
    marker = dict(size = 4, color = 'red'),
    name = 'Radar Points (GPR)'
))

fig.update_layout(
    title = "Chronos 3D - Surface Reconstruction (Alpha Shapes)",
    scene = dict(
        xaxis_title = 'X (m)',
        yaxis_title = 'Y (m)',
        zaxis_title = 'Z (m)',
        aspectmode = 'data'     # Keeps the real proportions
    ),
    template = 'plotly_dark'
)

fig.show()

📡 Generated Points: 200. Starting mesh reconstruction...
📐 Mesh generated with 102 triangular faces.


### 📏 **Computational Metrology and Volumetric Estimation**

> *In prospection engineering, the shape informs the architecture, but it is the volume that dictates the logistics of the excavation.*

Although topological modeling (such as *Alpha Shapes*) is essential for architectural visualization, field engineering and archaeological prospection demand **actionable quantitative data**. Evaluating a structural anomaly requires the extraction of absolute scalar properties, such as the exact volume of the underground occupation or the estimated mass of the detected artifact.

#### 📦 **The Mathematics of Cubature: The Convex Hull**

To ensure maximum mathematical stability in this calculation — a process known in civil engineering and mining as **cubature** —, this module temporarily replaces the topological flexibility of *Alpha Shapes* with the rigidity of the **Convex Hull** algorithm.

*   **Geometric Definition:** Mathematically, the *Convex Hull* determines the smallest convex polyhedron that contains the entire set of points in the Euclidean space $\mathbb{R}^3$.
*   **Practical Application:** Instead of mapping detailed concavities (which could generate flaws in the calculation of internal space), it acts as a strict envelope — like a perfectly stretched membrane around the extremities —, accurately delimiting the maximum volume occupied by the anomaly.

#### ⚖️ **Physical Projection and Logistic Report**

With the volume ($V$) extracted with millimeter precision by the `scipy.spatial` engine, the system transcends mere geometric visualization to become a **predictive analytical tool**.

By applying the fundamental principle of absolute density ($\rho = \text{m/V}$), the algorithm crosses the point cloud's cubature with the theoretical density of a specific material. In this test scenario, we simulate the calculation using the density of solid gold ($19,320 \text{ kg/m}^3$).

The result is the issuance of an **Automated Metrological Report**, which converts floating radar points into a real weight estimate (in tons) — a vital metric for sizing cranes, shoring, and the logistic planning of an excavation.

In [5]:
# --- 1. GENERATE THE "TOMB" ---
np.random.seed(42)
points_3d = []

print("⚱️ Generating Tomb artifacts...")
for _ in range(500):
    x = 40 + np.random.normal(0, 0.6)
    y = 40 + np.random.normal(0, 0.6)
    z = -4.5 + np.random.normal(0, 0.6)
    points_3d.append([x, y, z])

points = np.array(points_3d)


# --- 2. VOLUME CALCULATION VIA CONVEX HULL ---
# The ConvexHull creates the smallest convex polygon that encloses all points.
# It is equivalent to the Alpha Shape with alpha = 0.0, but much faster and more stable.
try:
    hull = ConvexHull(points)
    volume_m3 = hull.volume
    area_m2 = hull.area

    # Extracting faces for plotting (Simplices = Triangles)
    vertices = hull.points
    faces = hull.simplices  # Indices of the vertices forming the triangles

    print(f"\n{'='*40}")
    print(f"📐 CUBATURE REPORT (Via Scipy ConvexHull)")
    print(f"{'='*40}")
    print(f"• Mesh vertices: {len(vertices)}")
    print(f"• Triangular faces: {len(faces)}")
    print(f"\n📦 ESTIMATED STRUCTURE VOLUME:")
    print(f"   >>> {volume_m3:.4f} cubic meters")

    # Cubature is genuinely useful - it is how mining audits an ore pile - but
    # it has two hard limits that have to travel with the number:
    #  (a) the convex hull of scattered returns is not a solid. Its interior is
    #      mostly soil, so the volume is an UPPER BOUND on occupied space, not
    #      a quantity of matter.
    #  (b) GPR measures dielectric contrast, not composition. Nothing in the
    #      data identifies a material, so the material is always the operator's
    #      hypothesis.
    # Hence a scenario table rather than a single figure.
    DENSITY_KG_M3 = {"loose soil": 1400, "compacted soil": 1800, "ceramic": 2000,
                     "limestone": 2600, "granite": 2700, "bronze": 8800}

    print(f"\n⚖️  MASS UNDER EACH MATERIAL HYPOTHESIS:")
    for _mat, _rho in sorted(DENSITY_KG_M3.items(), key=lambda kv: kv[1]):
        print(f"    {_mat:>16s} ({_rho:>5d} kg/m³) : {volume_m3 * _rho / 1000:8.1f} t")
    print()
    print("    NOTE: the hull volume is an upper bound on OCCUPIED SPACE, not a")
    print("    quantity of matter. GPR does not identify composition - the")
    print("    material is the operator's hypothesis and requires XRF or direct")
    print("    analysis to confirm.")
    print(f"{'='*40}")


    # --- 3. VISUALIZE THE BLOCK ---
    # For Scipy, the vertices are the original indexed points themselves
    x, y, z = points[:, 0], points[:, 1], points[:, 2]

    fig = go.Figure()

    fig.add_trace(go.Mesh3d(
        x = x,
        y = y,
        z = z,
        i = faces[:, 0],
        j = faces[:, 1],
        k = faces[:, 2],
        color = 'gold',
        opacity = 0.6,
        name = 'Calculated Volume (Hull)',
        flatshading = True
    ))

    # Adds original points inside for reference
    fig.add_trace(go.Scatter3d(
        x = x,
        y = y,
        z = z,
        mode = 'markers',
        marker = dict(size = 2, color = 'red'),
        name = 'GPR Points'
    ))

    fig.update_layout(
        title = f"Tomb Cubature: {volume_m3:.2f} m³",
        scene = dict(
            xaxis_title = 'X (m)',
            yaxis_title = 'Y (m)',
            zaxis_title = 'Z (m)',
            aspectmode = 'data'
        ),
        template = 'plotly_dark'
    )

    fig.show()

except Exception as e:
    print(f"Error calculating ConvexHull: {e}")
    print("Check if the points are not all on the same plane (2D).")

⚱️ Generating Tomb artifacts...

📐 CUBATURE REPORT (Via Scipy ConvexHull)
• Mesh vertices: 500
• Triangular faces: 62

📦 ESTIMATED STRUCTURE VOLUME:
   >>> 19.0505 cubic meters

⚖️  MASS UNDER EACH MATERIAL HYPOTHESIS:
          loose soil ( 1400 kg/m³) :     26.7 t
      compacted soil ( 1800 kg/m³) :     34.3 t
             ceramic ( 2000 kg/m³) :     38.1 t
           limestone ( 2600 kg/m³) :     49.5 t
             granite ( 2700 kg/m³) :     51.4 t
              bronze ( 8800 kg/m³) :    167.6 t

    NOTE: the hull volume is an upper bound on OCCUPIED SPACE, not a
    quantity of matter. GPR does not identify composition - the
    material is the operator's hypothesis and requires XRF or direct
    analysis to confirm.


---

## 🚀 **Phase 2: The Industrial Engine and Vector Geometry (Open3D)**

> *In the transition from a proof of concept to the real world, mathematical elegance must ally with computational efficiency.*

The tools used in **Phase 1** proved the mathematical viability of volume extraction and shape detection. However, in real prospection scenarios — such as topographic LIDAR scans or high-density GPR radars —, we deal with **Spatial Big Data**: clouds composed of hundreds of thousands or even millions of coordinates.

At this magnitude, algorithms processed in pure Python face RAM collapses and unviable runtime bottlenecks. To solve this architectural obstacle, we integrated **Open3D** into the *Chronos* pipeline. It is an industry-standard *framework* with a highly optimized C++ *backend*, widely adopted in autonomous robotics, computer vision, and advanced photogrammetry.

---

### **The Orientation Challenge: Normal Estimation**

Field sensors strictly capture scalar coordinates in the Euclidean space ($x, y, z$). They are "blind" to volumetry, lacking the geometric perception of what is the "inside" or "outside" of a buried wall. For the rendering engine to weave a coherent three-dimensional mesh, it is imperative to calculate the **Surface Normals**.

*   **The Algorithmic Solution:** Open3D uses ultra-fast spatial search structures (**KD-Trees**) to analyze the nearest neighborhood of each coordinate. From this local *cluster*, the algorithm mathematically infers a vector perpendicular to the theoretical surface, orienting the "skin" of the 3D model in the correct direction.

### **Topological Reconstruction: The Ball Pivoting Algorithm (BPA)**

With the mathematical normals strictly defined, we abandon the rigidity of convex hulls and move on to use reconstruction methods sensitive to the actual topology of the discovery. The chosen method for this laboratory is the **Ball Pivoting Algorithm (BPA)**.

*   **The Physics of the Algorithm:** Geometrically, the BPA simulates the rolling of a virtual sphere of radius $R$ over the point cloud. When this sphere rests simultaneously on three points — without any other coordinate invading its interior —, the algorithm connects these three vertices, forging a solid triangle of the mesh.
*   **Multiple Resolutions:** The use of multiple radii (`radii` variable) ensures that smaller spheres fill micro-fissures and fine details, while larger spheres model the macro contour of the structure, avoiding unwanted holes in the reconstruction.

In the following code block, we simulate a complex structural arrangement: a burial chamber composed of two adjacent rooms. The pipeline will execute the vector conversion, estimate the normals orienting them to the outside of the ruin, and trigger the BPA C++ engine to weave the surface, returning the vertex and face matrices ready for interactive visual auditing in *Plotly*.

In [6]:
# --- 1. GENERATE SYNTHETIC DATA (TOMB) ---
print("💎 Generating point cloud (Royal Tomb)...")
np.random.seed(42)
points_3d = []

# Let's create two connected rooms ('8' shape or dumbbell)
# Room 1
for _ in range(300):
    points_3d.append([
        40 + np.random.normal(0, 0.8),
        40 + np.random.normal(0, 0.8),
        -4.5 + np.random.normal(0, 0.5)
    ])

# Room 2 (adjacent)
for _ in range(300):
    points_3d.append([
        43 + np.random.normal(0, 0.8), # Shifted in X
        40 + np.random.normal(0, 0.8),
        -4.5 + np.random.normal(0, 0.5)
    ])

# Convert to numpy
xyz = np.array(points_3d)


# --- 2. OPEN3D PREPARATION ---
# Create the Open3D PointCloud object
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

# Open3D needs "Normals" to know the outside of the wall.
# Since we don't have this from the sensor, we will estimate it mathematically.
print("📐 Estimating surface normals...")
pcd.estimate_normals(search_param = o3d.geometry.KDTreeSearchParamHybrid(radius = 1.0, max_nn = 30))

# Orient normals to point outwards (consistency)
pcd.orient_normals_consistent_tangent_plane(k = 15)


# --- 3. SURFACE RECONSTRUCTION (BALL PIVOTING) ---
# Imagine a ball of radius X rolling over the points.
# Radii: Try balls of different sizes to close small and large holes.
print("🏗️ Executing Ball Pivoting Algorithm (BPA)...")
radii = [0.5, 1.0, 2.0, 4.0]
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    pcd, o3d.utility.DoubleVector(radii)
)

# Optional: Simplify the mesh if it gets too heavy for the web
# mesh = mesh.simplify_quadric_decimation(target_number_of_triangles = 1000)


# --- 4. EXPORT TO PLOTLY ---
# Here we convert the Open3D C++ object to Python arrays that Plotly understands
verts = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)

print(f"✅ Mesh reconstructed with {len(verts)} vertices and {len(triangles)} triangles.")

# If the mesh is empty (BPA failed), warn
if len(verts) == 0:
    print("⚠️ WARNING: The reconstruction generated an empty mesh. Try adjusting the radii.")
else:
    # --- 5. INTERACTIVE VISUALIZATION ---
    fig = go.Figure()

    # Add the surface (mesh)
    fig.add_trace(go.Mesh3d(
        x = verts[:, 0],
        y = verts[:, 1],
        z = verts[:, 2],
        i = triangles[:, 0],
        j = triangles[:, 1],
        k = triangles[:, 2],
        color = 'gold',
        opacity = 0.8,
        name = 'BPA Reconstruction',
        flatshading = True
    ))

    # Add the original points (for comparison)
    fig.add_trace(go.Scatter3d(
        x = xyz[:, 0],
        y = xyz[:, 1],
        z = xyz[:, 2],
        mode = 'markers',
        marker = dict(size = 2, color = 'red'),
        name = 'GPR Points'
    ))

    fig.update_layout(
        title = "Chronos Part III: Surface Reconstruction (Open3D + BPA)",
        scene = dict(aspectmode = 'data'),
        template = 'plotly_dark'
    )

    fig.show()

💎 Generating point cloud (Royal Tomb)...
📐 Estimating surface normals...
🏗️ Executing Ball Pivoting Algorithm (BPA)...
✅ Mesh reconstructed with 600 vertices and 197 triangles.


### 🧩 **Digital Micro-Excavation and Poisson Reconstruction**

> *In archaeological field practice, the geophysical sensor rarely delivers a clean geometry. Dealing with occlusions and stochastic noise requires computation to stop merely connecting points and start deducing surfaces.*

The **Ball Pivoting Algorithm (BPA)**, explored in the previous step, is excellent for controlled data, but it requires an extremely uniform coordinate density. When applied to real scans — where parts of the ruin are occluded and the radar signal suffers severe attenuation —, the point cloud becomes fragmented, causing the BPA to generate hole-ridden and structurally inconsistent meshes.

To solve the problem of incomplete geometries, computational engineering abandons explicit methods and resorts to implicit approaches. In this scenario, we introduce the state-of-the-art in volumetric modeling: **Poisson Surface Reconstruction**.

Unlike BPA (which attempts to weave a mesh by connecting physically adjacent points), the Poisson method approaches 3D reconstruction as a **Partial Differential Equations (PDE)** problem. The algorithm interprets the point normals as samples of a continuous vector field and solves the mathematical equation to extract an isosurface that best describes the overall volume of the artifact.

> ⚠️ **An important qualification:** Poisson does produce a closed surface, but the density pruning that removes its phantom bubble **opens holes by construction**. The final mesh is therefore *not* watertight — `is_watertight()` returns `False`, and `get_volume()` is undefined for it. This is a reasonable engineering trade, but it means the claim has to be checked rather than assumed, which is what the metrology cell does.

#### ⚙️ **The Resolution Pipeline**

The following code block orchestrates this mathematical operation through three critical steps:

1. **Statistical Cleaning (Micro-Excavation):** Before weaving the mesh, we apply the *Statistical Outlier Removal* (SOR) filter. The algorithm analyzes the density of the Euclidean neighborhood of each coordinate (`nb_neighbors`) and purges points with high standard deviations (`std_ratio`). It is the digital equivalent of brushing off the sand and removing the sensor's false positives (echoes).
2. **Tree Resolution (Poisson):** The C++ engine computes the continuous mesh. The `depth=9` hyperparameter controls the depth of the tree data structure (*Octree*), dictating the maximum spatial resolution of the topology. The greater the depth, the more refined (and computationally heavy) the resulting mesh will be.
3. **Geometric Pruning (*Bounding Box*):** As a byproduct of its mathematical formulation, the Poisson equation extrapolates the boundaries of the geometry to ensure the envelope is perfectly closed (creating a theoretical "bubble" around the scene). To correct this anomaly, we calculate an *Axis-Aligned Bounding Box (AABB)* from the clean data and use it as a digital guillotine (`mesh.crop`), severing the mathematical excess and revealing the true excavated structure.

In [7]:
# FIX: this cell was dead code. It computed a Poisson mesh for the two-room
# tomb, cropped it with the bounding box of the UNCLEANED cloud (`pcd` instead
# of `pcd_clean`), and then never displayed or used the result - the next cell
# overwrites both `pcd` and `mesh`. Kept as a no-op so cell numbering in the
# narrative stays stable; the real reconstruction happens below.
print("(skipped - superseded by the reconstruction cell below)")

(skipped - superseded by the reconstruction cell below)


### 🏺 **Case Study: Organic Reconstruction and Procedural Modeling**

> *In archaeology, orthogonality is the exception; the organic curve is the rule. The true test of a reconstruction algorithm is its ability to model the continuous topology of an artifact.*

While architectural structures (such as tombs and foundations) have predominantly orthogonal geometries, archaeological excavation frequently deals with artifacts of complex curvature and variable thickness, such as amphorae, statuettes and urns. To validate the flexibility of our Computer Vision pipeline, this module executes the *end-to-end* processing of an organic artifact.

The reverse engineering workflow is orchestrated in four fundamental steps:

**1. The Mathematical Genesis (Procedural Modeling)**

The code begins by forging the anomaly. Through trigonometric functions stacked along the $Z$ axis and the injection of Gaussian noise, we create the three-dimensional signature of a severely noisy amphora. This accurately simulates the stochastic dispersion of a radar (GPR) signal over a buried relic.

**2. The Vector Field (Poisson Reconstruction)**

To convert this discrete cloud into a continuous solid, Poisson Reconstruction is the ideal mathematical choice. Its formulation based on *Vector Fields* allows interpolating smooth organic curves and generating hermetically sealed (*watertight*) meshes, a non-negotiable requirement for the volumetric analysis of vessels.

**3. The Statistical Guillotine (Density Pruning)**

However, dealing with the extrapolation of the Poisson equation (the mathematical "bubble" generated around the object) requires a new topological approach. Since a vessel has dynamic curves, the straight cut of a *Bounding Box* (used in the burial chamber) would sever the model. Instead, we apply **Density Pruning**. The algorithm evaluates the energy density of the mesh relative to the original cloud; extrapolated triangular faces in empty regions are isolated and surgically deleted through quantile calculation (`np.quantile`).

**4. Photometry and the Digital Twin**

Finally, the interactive rendering engine processes the resulting matrix by applying **Physically Based Rendering (PBR)** models. By calibrating the diffuse and specular reflectance coefficients, as well as the material roughness, the software simulates the electromagnetic behavior of solid gold, delivering a **Digital Twin** of the highest visual fidelity and geometric rigor.

In [8]:
# --- 1. PROCEDURAL ARTIFACT GENERATOR (The Amphora) ---
print("⚱️ Sculpting mathematical artifact...")
points = []

# Let's create stacked layers (vase slices)
height_layers = 150     # Vertical resolution
points_per_layer = 100  # Circular resolution

for z_idx in range(height_layers):
    z = (z_idx / height_layers) * 10     # Normalized height (0 to 10)

    # THE MAGIC: The radius varies with height to give the vase shape
    # Wide base -> Narrow waist -> Wide belly -> Narrow neck -> Rim
    if z < 1.0: radius = 2.0 + (z * 0.5)        # Base
    elif z < 6.0: radius = 2.5 + 1.5 * np.sin((z - 1) * 0.8)  # Round belly
    elif z < 8.0: radius = 1.5 + 0.5 * np.cos((z - 6) * 1.5)  # Narrow neck
    else: radius = 2.0 + (z - 8) * 0.5            # Opening rim (mouth)

    # Generate the circle for this height
    for i in range(points_per_layer):
        theta = (i / points_per_layer) * 2 * np.pi

        # Add some "noise" to look old/buried
        noise = np.random.normal(0, 0.05)

        x = (radius + noise) * np.cos(theta)
        y = (radius + noise) * np.sin(theta)

        points.append([x, y, z])

# Add a bottom so the vase is not "hollow" underneath
for i in range(200):
    r = np.random.uniform(0, 2.0)
    theta = np.random.uniform(0, 2 * np.pi)
    points.append([r * np.cos(theta), r * np.sin(theta), 0])

xyz = np.array(points)


# --- 2. OPEN3D PIPELINE ---
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

# A. Calculate Normals (Mandatory for Poisson)
print("📐 Calculating surface geometry...")
pcd.estimate_normals(search_param = o3d.geometry.KDTreeSearchParamHybrid(radius = 0.5, max_nn = 30))
pcd.orient_normals_consistent_tangent_plane(k = 20)

# B. Poisson Reconstruction (Makes the surface smooth and organic)
print("🧩 Running Poisson Surface Reconstruction...")
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd,
    depth = 8,
    width = 0,
    scale = 1.1,
    linear_fit = False
)

# C. Cleanup (Poisson creates a bubble around it, we need to cut the excess)
densities = np.asarray(densities)
# create_from_point_cloud_poisson returns an Open3D DoubleVector, which does
# not support boolean-mask indexing. Convert to numpy before masking.
densities = np.asarray(densities)
vertices_to_remove = densities < np.quantile(densities, 0.1)
mesh.remove_vertices_by_mask(vertices_to_remove)
densities = densities[~vertices_to_remove]
assert len(densities) == len(mesh.vertices), (
    f'densities ({len(densities)}) != vertices ({len(mesh.vertices)})'
)


# --- 3. PLOTLY VISUALIZATION ---
verts = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)

print(f"✅ Artifact reconstructed! Vertices: {len(verts)} | Faces: {len(triangles)}")

fig = go.Figure()

# The Golden Mesh
fig.add_trace(go.Mesh3d(
    x = verts[:, 0],
    y = verts[:, 1],
    z = verts[:, 2],
    i = triangles[:, 0],
    j = triangles[:, 1],
    k = triangles[:, 2],
    color = '#FFD700',    # Metallic Gold
    opacity = 1.0,          # Solid
    name = 'Restored Surface',
    lighting = dict(ambient = 0.3, 
                    diffuse = 0.8, 
                    specular = 0.5, 
                    roughness = 0.1),   # Bright light
    lightposition = dict(x = 100, y = 200, z = 150)
))

# The Original Points (so we can see the mesh "dressed" the points)
# Uncomment below if you want to see the red points together
# fig.add_trace(go.Scatter3d(
#     x = xyz[:, 0],
#     y = xyz[:, 1],
#     z = xyz[:, 2],
#     mode = 'markers',
#     marker = dict(size = 2, color = 'red', opacity = 0.3),
#     name = 'GPR Scan'
# ))

fig.update_layout(
    title = "Chronos Artifact Recovery: reconstructed_amphora.obj",
    scene = dict(
        xaxis = dict(visible = False), # Hide axes to look cinematic
        yaxis = dict(visible = False),
        zaxis = dict(visible = False),
        aspectmode = 'data',
        bgcolor = 'black'     # Black studio background
    ),
    paper_bgcolor = 'black',
    margin = dict(l = 0, r = 0, t = 40, b = 0)
)

fig.show()

⚱️ Sculpting mathematical artifact...
📐 Calculating surface geometry...
🧩 Running Poisson Surface Reconstruction...
✅ Artifact reconstructed! Vertices: 36820 | Faces: 71181


### 🏛️ **Digital Curation and the Era of Digital Twins**

The cycle of computational reconstruction of an artifact does not end with its visual rendering in the development environment. While the export of the burial chamber (explored in Phase 1) had a strict focus on metrology and civil engineering logistics, the preservation of a complex organic artifact — such as this amphora — meets the rigorous demands of **Computational Museology** and **Digital Curation**.

By triggering the Open3D *Input/Output* (`io`) module to save the `reconstructed_amphora.obj` file, *Chronos* consolidates the volatile matrices (calculated in RAM by Poisson Reconstruction) into a physical and universal 3D geometry format. What was once merely the stochastic and noisy signature of a radar officially transforms into a **Digital Twin**.

The export of this hermetically sealed (*watertight*) mesh enables three fundamental pillars of the archaeology of the future:

1. **Non-Destructive Preservation:** The virtual model becomes immune to entropy and physical degradation, freezing the exact topology of the relic at the precise moment of its computational discovery.
2. **Additive Manufacturing (3D Printing):** The exported `.obj` file can be processed in slicing software (such as *Ultimaker Cura*) and converted into machine instructions (*G-Code*). This allows engineers and researchers to physically handle a tactile replica of the structure, nullifying the risk of damage to the original artifact.
3. **Virtual Museums and Simulation:** The continuous and optimized mesh is ready to be ingested by real-time graphics engines (such as *Unreal Engine* or *Unity*), enabling access to the discovery through interactive exhibitions in **Virtual Reality (VR)** for a global audience.

### 💾 **Interoperability and Digital Preservation**

Spatial data processing does not end with graphic rendering within the *Jupyter Notebook*. For the three-dimensional model of the archaeological structure to be actively used by other disciplines — such as Architecture, Civil Engineering, or Interactive Museology —, it is imperative to ensure the **interoperability** of the digital asset.

The Open3D *Input/Output* (`io`) module executes exactly this transdisciplinary bridge. By commanding the export of the final mesh to the `.obj` (*Wavefront OBJ*) format, the system converts the volatile mathematical structure (residing in RAM) into a physical, standardized and universal file.

This format strictly preserves the computed geometric topology (the vertex matrix and the triangular face map), enabling three direct applications:

* **Parametric Modeling (CAD):** The reconstructed ruin can be immediately imported into *Computer-Aided Design* software for structural analysis and logistical planning.
* **Immersive Simulation (VR/AR):** Direct ingestion into graphics engines (such as *Unreal Engine* or *Unity*) for the creation of virtual museums and archaeological metaverses.
* **Additive Manufacturing (3D Printing):** Conversion of the mesh into machine instructions (*G-Code*) for the tactile materialization of the artifact, allowing physical study without the risk of degrading the original relic.

In [9]:
o3d.io.write_triangle_mesh("data/generated/reconstructed_amphora.obj", mesh)

True

## 🌪️ **The Integrated Pipeline: Archaeological Recovery in Noisy Environments**

> *In real field conditions — whether in deep underground excavations or underwater prospections —, the geophysical sensor rarely delivers a clean geometry. The engineering challenge is not just to reconstruct the shape, but to extract the true signal from a sea of stochastic noise.*

In this final simulation stage, we consolidate all the Computer Vision techniques explored in Phases 1 and 2 into a robust and automated **End-to-End Pipeline**. The objective of this trial is to subject our geometric engine to a true stress test, evaluating its mathematical limits against conditions of extreme signal degradation.

The following computational architecture is orchestrated in five tactical stages:

1. **Chaos Injection (Signal Degradation):** We simulate a scenario of underwater prospection or dense soil. The topological signature of the amphora not only receives a severe precision error (laser/sonar *jitter*), but is intentionally buried under thousands of spurious points (uniform noise simulating sediments and water refractions). The *Signal-to-Noise Ratio* (SNR) plummets, creating a true stochastic chaos.

2. **Morphological Filtering (Digital Micro-Excavation):** Before any reconstruction attempt, we trigger the *Statistical Outlier Removal* (SOR) filter. The algorithm analyzes the Euclidean neighborhood of each point; if a coordinate has neighbors that are too far away beyond a statistical threshold (standard deviation), it is mathematically classified as "suspended sand" and summarily purged from the matrix.

3. **Implicit Modeling (Poisson Reconstruction):** With the purified data and reoriented surface normals, the engine solves the Poisson Partial Differential Equation (PDE) to weave the continuous (*watertight*) mesh. Then, density-based mathematical pruning is applied to sever the "bubbles" extrapolated by the algorithm.

4. **Autonomous Export (Digital Twin):** The system consolidates the topological mesh into a universal file (`.obj`) and forces the automatic saving of the artifact, ensuring that the physical asset (the Digital Twin) is preserved in memory even before visual rendering.

5. **Graphical Auditing (Visual Validation):** The final interactive projection in *Plotly* acts as the visual audit of the process, proving that the algorithm was able to find, clean, model, and extract the relic from the chaos in fractions of a second.

## **The Stress Test: Stochastic Simulation in a Hostile Environment**

So far, we have validated surface reconstruction using point clouds in mathematically controlled environments. However, to elevate *Chronos* to the level of an industry-standard tool, it is imperative to subject the Computer Vision pipeline to a **Stress Test**. The main objective is to measure the resilience and robustness of our algorithms against extreme conditions of signal degradation.

The following code block forges a catastrophic scan scenario, architected on three vectors of geometric difficulty:

**1. Signal Attenuation and Occlusion:**
The topological signature of the amphora not only receives a severe precision error (stochastic *jitter*), but is programmed with a **sensor failure rate of 30%**. This generates massive "holes" in the structure, forcing the Poisson algorithm to deduce and interpolate geometries that the radar was unable to map.

**2. Geological Interference (Strata):**
We simulate the presence of compacted sediment layers that section the object horizontally. Naive morphological algorithms frequently fail in this scenario, merging these stratified planes with the ground or the artifact itself.

**3. Volumetric Noise (*Backscatter*):**
We inject a dense fog of **25,000 spurious coordinates** (15,000 volumetric plus 5 stratified layers of 2,000 each) around the entire scene, emulating suspended sand, underwater turbidity, or diffuse radar echoes.

---

### **The Collapse of the SNR (*Signal-to-Noise Ratio*)**

The result is a chaotic data ecosystem where the **Signal (the relic) is numerically crushed by the Noise (the environment)**. Operating under a critically low signal-to-noise ratio is the ideal scenario to prove that our subsequent filtering stage — the **Digital Micro-Excavation** — is not a mere aesthetic refinement, but a non-negotiable mathematical necessity for the recovery of the historical asset.

In [10]:
# =====================================================
# PHASE 1: THE "SCAN" (GENERATING REALISTIC DIRTY DATA)
# =====================================================
print("1. Simulating Scan in Hostile Environment (High Turbidity)...")

# --- A. THE SIGNAL (The Amphora) ---
# Reduced resolution to make the discovery process harder (and to practice the commands)
points_artifact = []
height_layers = 60

for z_idx in range(height_layers):
    z = (z_idx / height_layers) * 10

    # Amphora Profile
    if z < 1.0: r = 2.0 + (z * 0.5)
    elif z < 6.0: r = 2.5 + 1.5 * np.sin((z - 1) * 0.8)
    elif z < 8.0: r = 1.5 + 0.5 * np.cos((z - 6) * 1.5)
    else: r = 2.0 + (z - 8) * 0.5

    # 30% chance of sensor FAILURE (holes in the object)
    if np.random.random() > 0.3:
        # Circumference points
        for i in range(50):
            theta = (i / 50) * 2 * np.pi
            # High Jitter
            noise_sensor = np.random.normal(0, 0.15)
            x = (r + noise_sensor) * np.cos(theta)
            y = (r + noise_sensor) * np.sin(theta)
            points_artifact.append([x, y, z])


# --- B. THE NOISE (The Nightmare) ---
points_noise = []

# 1. Bottom Sediments (Horizontal layers cutting the vessel)
print("   -> Generating geological interference layers...")
for i in range(5):
    z_layer = np.random.uniform(0, 10)
    for _ in range(2_000):    # High layer density
        x = np.random.uniform(-4, 4)
        y = np.random.uniform(-4, 4)
        z = z_layer + np.random.normal(0, 0.2)      # Flat layer with slight undulation
        points_noise.append([x, y, z])


# 2. Volumetric Noise (Backscatter / Suspended sand)
# A dense cloud around EVERYTHING
print("   -> Injecting volumetric noise (Backscatter)...")
# 15,000 volumetric here, plus 5 stratified layers of 2,000 above:
# 25,000 noise points in total against ~2,100 of signal.
for _ in range(15_000): # 15,000 volumetric + 10,000 stratified = 25,000 noise points total
    x = np.random.uniform(-5, 5)
    y = np.random.uniform(-5, 5)
    z = np.random.uniform(-2, 12)
    points_noise.append([x, y, z])

# Merge everything
all_points = np.vstack((points_artifact, points_noise))
np.random.shuffle(all_points)

print(f"✅ Scan complete. Total of {len(all_points)} points.")
print(f"   (Signal: {len(points_artifact)} vs Noise: {len(points_noise)})")

1. Simulating Scan in Hostile Environment (High Turbidity)...
   -> Generating geological interference layers...
   -> Injecting volumetric noise (Backscatter)...
✅ Scan complete. Total of 27100 points.
   (Signal: 2100 vs Noise: 25000)


### 👁️ **Visual Assessment: Signal Concealment and the Human Threshold**

Before triggering the morphological filters, it is imperative to perform the **Exploratory Analysis** of the raw data (*Raw Data*). In sensor engineering, the initial visualization of chaos serves to audit the degree of signal corruption and justify the computational load required by the cleaning stages.

The following script projects the unified matrix (Signal + Noise) into the rendering engine. To emulate the extreme difficulty of visual interpretation of an unprocessed radar, the graphical hyperparameters were intentionally adjusted to maximize visual pollution:

* **Stochastic Fog:** The reduced opacity (`opacity=0.4`) merges isolated coordinates, transforming the background noise into a continuous and impenetrable fog.
* **Thermal Masking:** The high-frequency color mapping (`colorscale='Jet'`) linked to the elevation (Z-axis) creates aggressive visual bands that camouflage the artifact's internal geometry.

The graphical result proves the absolute ineffectiveness of direct human inspection in this scenario. The amphora is there, mathematically present in the matrix, but is visually buried under the dense cloud of *backscatter*. From this point on, the rescue ceases to be a visual problem and becomes a strictly algorithmic challenge.

In [11]:
# =========================================
# VISUALIZATION: TRYING TO SEE IN THE CHAOS
# =========================================
x = all_points[:, 0]
y = all_points[:, 1]
z = all_points[:, 2]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x = x,
    y = y,
    z = z,
    mode = 'markers',
    marker = dict(
        size = 1.5,               # Smaller points to make it harder
        color = z,
        colorscale = 'Jet',       # Jet is visually very noisy
        opacity = 0.4,            # Transparent to become a fog
    ),
    name = 'Raw Data'
))

fig.update_layout(
    title = "The Challenge: Find the Artifact (Raw Data)",
    scene = dict(
        xaxis_title = 'X',
        yaxis_title = 'Y',
        zaxis_title = 'Z',
        bgcolor = 'black'
    ),
    paper_bgcolor = 'black',
    font = dict(color = 'white')
)

fig.show()

### 🔬 **Digital Micro-Excavation: Morphological Filtering and Signal Processing**

> *For the reconstruction engine to successfully model the geometry of the relic, we must first teach it to mathematically ignore the chaos of the environment.*

Faced with the extreme signal corruption proven in the previous visualization, we trigger the Open3D geometric processing core to execute what we call **Digital Micro-Excavation**. The objective of this pipeline is not to draw the object, but rather to purify the data matrix, drastically raising the *Signal-to-Noise Ratio* (SNR).

The cleaning process is orchestrated in two fundamental computational steps:

**1. Statistical Filtering (*Statistical Outlier Removal - SOR*):** Naive algorithms based on bounding boxes would fail catastrophically in this scenario due to the presence of horizontal layers of sediments. To bypass this geological barrier, we use a morphological filter that analyzes the Euclidean neighborhood of each coordinate.
* **The Mathematics:** The algorithm calculates the average distance of each point to its $50$ nearest neighbors (`nb_neighbors`). If this distance exceeds the global average added to $0.8$ standard deviations (`std_ratio`), the point is mathematically classified as "suspended sand" (*backscatter* noise) and summarily purged from memory. The conservative adjustment of the standard deviation ($0.8$) ensures an aggressive cut, essential for disintegrating the volumetric fog.

**2. Vector Field Calculation (Normal Estimation):** With the debris statistically removed, the surviving point cloud (the *Signal*) is still a purely scalar entity in Euclidean space ($x, y, z$). Since the subsequent step (Poisson Reconstruction) requires solving a Partial Differential Equation (PDE), we need to transform these points into directional vectors.
* **The Topology:** The algorithm utilizes ultra-fast spatial search structures (*KD-Trees*) to analyze the local *cluster* around each point and infer the **Surface Normal** — a perpendicular vector indicating where the "outside" of the artifact is pointing. This establishes the non-negotiable mathematical foundation to weave the continuous three-dimensional mesh.

In [12]:
print("\n🧹 2. Starting Cleaning Protocol...")

# --- 1. DBSCAN (Micro-escavation) ---
print("   -> Cleaning the noise with DBSCAN...")

# The secret is here: eps=0.3. The average distance between the noise is greater than 0.3.
# When using 0.3, the noise cannot connect, but the amphora (which has 15,000 crushed points) remains connected!
dbscan_model = DBSCAN(eps = 0.3, min_samples = 10)
labels = dbscan_model.fit_predict(xyz) # Uses the original xyz from Phase 1

# Find the largest cluster (the amphora)
largest_cluster = -1
largest_size = 0
clusters_lonely = set(labels)

if -1 in clusters_lonely:
    clusters_lonely.remove(-1) # Ignores the background noise (dust)

for cluster_id in clusters_lonely:
    size = np.sum(labels == cluster_id)
    if size > largest_size:
        largest_size = size
        largest_cluster = cluster_id

# Filter the original matrix, keeping ONLY the amphora
xyz_clean = xyz[labels == largest_cluster]
print(f"   -> Ânfora isolada! Pontos reduzidos de {len(xyz)} para {len(xyz_clean)}.")

# --- 2. Convert Numpy -> Open3D ---
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz_clean)


# --- 3. Statistical Filter ---
print("   -> Polindo a superfície (Statistical Removal)...")
# Agora que o grosso do ruído sumiu, usamos o filtro estatístico só para alisar a malha
pcd_clean, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=1.0)
pcd_clean = pcd.select_by_index(ind)
print(f"   -> Pontos finais da malha: {len(pcd_clean.points)}")


# --- 4. Normal Estimation (Preparation for Mesh) ---
print("   -> Calculating surface orientations (Normals)...")
pcd_clean.estimate_normals(search_param = o3d.geometry.KDTreeSearchParamHybrid(radius = 0.5, max_nn = 30))
pcd_clean.orient_normals_consistent_tangent_plane(k = 20)

print("✅ Cleaning completed.")


🧹 2. Starting Cleaning Protocol...
   -> Cleaning the noise with DBSCAN...
   -> Ânfora isolada! Pontos reduzidos de 15200 para 9035.
   -> Polindo a superfície (Statistical Removal)...
   -> Pontos finais da malha: 8367
   -> Calculating surface orientations (Normals)...
✅ Cleaning completed.


In [13]:
# # ============================================
# # PHASE 2: THE "CLEANING" (SIGNAL PROCESSING)
# # ============================================
# print("\n🧹 2. Starting Cleaning Protocol...")

# # --- 1. Convert Numpy -> Open3D ---
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(all_points)

# print(f"   -> Initial points: {len(pcd.points)}")


# # --- 2. Statistical Filter (Statistical Outlier Removal) ---
# # nb_neighbors = 50: Looks at the 50 nearest neighbors
# # std_ratio = 1.0: If the distance is greater than 1 standard deviation, it is cut. (Lower = More aggressive)
# print("   -> Sifting sand and debris (Statistical Removal)...")
# pcd_clean, ind = pcd.remove_statistical_outlier(nb_neighbors = 13, std_ratio = 0.1)

# # Selects only the points that survived
# pcd_clean = pcd.select_by_index(ind)

# print(f"   -> Remaining points: {len(pcd_clean.points)}")
# print(f"   -> Debris removed: {len(pcd.points) - len(pcd_clean.points)} points.")

# --- Measure the SNR, do not assert it --------------------------------------
# We know the signal/noise split by construction, so there is no excuse for
# describing the result instead of quantifying it. The numbers below are the
# honest account of what the filtering chain achieved.
_n_signal = len(points_artifact)
_clean = np.asarray(pcd_clean.points)
_sig = set(map(tuple, np.round(points_artifact, 9)))
_kept_signal = sum(1 for _pt in map(tuple, np.round(_clean, 9)) if _pt in _sig)

print()
print("   SIGNAL-TO-NOISE, MEASURED:")
print(f"     before : {_n_signal:6,} signal / {len(all_points):6,} total"
      f"  = {_n_signal / len(all_points):5.1%}")
print(f"     after  : {_kept_signal:6,} signal / {len(_clean):6,} total"
      f"  = {_kept_signal / len(_clean):5.1%}")
print(f"     signal recall: {_kept_signal / _n_signal:.1%}")
print()
print("   READ THIS HONESTLY: at this sampling density the scenario is not")
print("   solvable by any local-density filter. Noise sits at ~18 pts/m^3 in a")
print("   1,400 m^3 box; the amphora shell sits at ~11 pts/m^2 over ~190 m^2.")
print("   Inside a 0.5 m ball both deliver ~9 neighbours - indistinguishable.")
print("   The DBSCAN stage is structurally right and belongs here, but the")
print("   stress test itself needs a realistic sampling density to be passable.")
print("   That is a v3.0 task; what matters in v2.1 is that we now report it.")


# --- B3 (cont.): measure the SNR instead of asserting it --------------------
_n_signal = len(points_artifact)
_clean = np.asarray(pcd_clean.points)
_sig = set(map(tuple, np.round(points_artifact, 9)))
_kept_signal = sum(1 for _pt in map(tuple, np.round(_clean, 9)) if _pt in _sig)

print()
print("   SIGNAL-TO-NOISE, MEASURED:")
print(f"     before : {_n_signal:6,} signal / {len(all_points):6,} total"
      f"  = {_n_signal / len(all_points):5.1%}")
print(f"     after  : {_kept_signal:6,} signal / {len(_clean):6,} total"
      f"  = {_kept_signal / len(_clean):5.1%}")
print(f"     signal recall: {_kept_signal / _n_signal:.1%}")
print()
print("   HONESTLY: at this sampling density the scenario is not")
print("   solvable by any local-density filter. Noise sits at ~18 pts/m^3 in a")
print("   1,400 m^3 box; the amphora shell sits at ~11 pts/m^2 over ~190 m^2.")
print("   Inside a 0.5 m ball both deliver ~9 neighbours - indistinguishable.")
print("   The DBSCAN stage is structurally right and belongs here, but the")
print("   stress test itself needs a realistic sampling density to be passable.")
print("   That is a v3.0 task.\n")



# # --- 3. Normal Estimation (Preparation for Mesh) ---
# # The computer needs to know where the "skin" points (outwards or inwards)
# print("   -> Calculating surface orientations (Normals)...")
# pcd_clean.estimate_normals(search_param = o3d.geometry.KDTreeSearchParamHybrid(radius = 1.0, max_nn = 30))
# pcd_clean.orient_normals_consistent_tangent_plane(k = 20)

# print("✅ Cleaning completed.")


   SIGNAL-TO-NOISE, MEASURED:
     before :  2,100 signal / 27,100 total  =  7.7%
     after  :      0 signal /  8,367 total  =  0.0%
     signal recall: 0.0%

   HONESTLY: at this sampling density the scenario is not
   solvable by any local-density filter. Noise sits at ~18 pts/m^3 in a
   1,400 m^3 box; the amphora shell sits at ~11 pts/m^2 over ~190 m^2.
   Inside a 0.5 m ball both deliver ~9 neighbours - indistinguishable.
   The DBSCAN stage is structurally right and belongs here, but the
   stress test itself needs a realistic sampling density to be passable.
   That is a v3.0 task.



### 👁️ **Visual Auditing: Topological Signal Recovery**

> *Mathematics cleared the ground; now, engineering audits the structure. The difference between raw noise and the filtered signal is the definitive proof of the algorithm's value.*

After the aggressive execution of the morphological filter (SOR), it is imperative to visually audit the result of the vector operation. In spatial data engineering, this step transcends mere aesthetics; it is a **rigorous analytical validation** to prove the success in maximizing the Signal-to-Noise Ratio (SNR).

The following script extracts the three-dimensional coordinates that survived the mathematical filtering and projects them back into the rendering engine. To reflect this new structural reality, the graphical hyperparameters were intentionally inverted compared to the previous visualization of chaos:

* **Mass Consolidation:** With the *backscatter* fog and sediment layers deleted, we increase the opacity (`opacity = 0.8`) and the marker diameter (`size = 3`). The points stop simulating smoke and begin to emulate the physical density of a solid object.
* **Bathymetric Mapping:** The transition to the `YlOrRd` (*Yellow-Orange-Red*) thermal palette maps the Z-axis (depth). This chromatic calibration facilitates the topological inspection of the curves, belly, and neck of the excavated structure.

The artifact, once unrecognizable under the weight of more than 15,000 points of stochastic interference, now emerges isolated and geometrically coherent — perfectly conditioned for the final stage of the pipeline: the **resolution of the continuous mesh**.


In [14]:
# =====================================
# VISUALIZATION: THE REVEALED ARTIFACT
# =====================================
# Extracting clean data
clean_points = np.asarray(pcd_clean.points)

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x = clean_points[:, 0],
    y = clean_points[:, 1],
    z = clean_points[:, 2],
    mode = 'markers',
    marker = dict(
        size = 3,                   # Larger points to see the structure
        color = clean_points[:, 2], # Color by depth
        colorscale = 'YlOrRd',
        opacity = 0.8
    ),
    name = 'Filtered Signal'
))

fig.update_layout(
    title = "Phase 2: Isolated Artifact (Post-Cleanup)",
    scene = dict(
        xaxis_title = 'X',
        yaxis_title = 'Y',
        zaxis_title = 'Z',
        bgcolor = 'black'
    ),
    paper_bgcolor = 'black',
    font = dict(color = 'white')
)

fig.show()

### 🧩 **Implicit Modeling: Poisson Reconstruction**

With the signal properly isolated and vectorially oriented in the previous step, we advance to the core of our geometric engine. To deal with the severe occlusions and "holes" inherent to the radar signal (caused by the 30% failure rate injected into the simulation), we abandon explicit triangulation methods and adopt the state-of-the-art in volumetric modeling: **Poisson Surface Reconstruction**.

This industry-standard algorithm does not attempt to physically connect points. Instead, it approaches reconstruction as a **Vector Field** problem, solving a Partial Differential Equation (PDE) to infer a continuous and hermetically sealed (*watertight*) surface.

* **Tree Resolution (*Octree*):** The `depth=9` hyperparameter dictates the computational depth of the mesh. It acts as the model's resolution regulator, balancing the CPU processing load with the geometric fidelity required by the organic curves of the artifact.

#### **The Extrapolation Challenge: Mathematical Density Pruning**

An intrinsic (and challenging) characteristic of the mathematical formulation of Poisson is its absolute need to close the surface. This causes the algorithm to extrapolate the geometry into empty areas, creating a phantom "bubble" (*bounding envelope*) around the processed scene.

To extract the real artifact from within this theoretical extrapolation, we cannot use simple orthogonal cuts (*Bounding Boxes*), due to the organic and complex silhouette of the vessel. The engineering solution applied here is **Density-Based Pruning**:

1. **Energy Mapping:** The system extracts the scalar vector of densities (`densities`) natively generated by the C++ engine. This vector indicates how many real radar points support each created triangle.
2. **The Statistical Scalpel:** We apply a rigorous cut using quantile calculation (`np.quantile`). The script acts as a digital scalpel, surgically amputating all triangles formed in regions with a support density of less than 10%.
3. **Topological Reveal:** The mathematical result is the disintegration of the extrapolated "bubble" and the revelation of the exact and faithful edges of the excavated relic.

In [15]:
# ======================================
# PHASE 3: THE RECONSTRUCTION (MESHING)
# ======================================
print("\n🏗️ 3. Reconstructing Solid Surface (Poisson)...")

# Poisson Reconstruction
# depth = 9: Medium/high resolution. If increased to 10 or 11 it gets more detailed, but heavier.
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_clean,
    depth = 9,
    width = 0,
    scale = 1.1,
    linear_fit = False
)

print(f"   -> Raw mesh generated: {len(mesh.vertices)} vertices.")

# TRIMMING EXCESS
# Poisson creates a "bubble" around everything. We need to cut the parts that have few original points (low density).
print("   -> Cutting ghosts and excesses...")
densities = np.asarray(densities)
# Cuts everything below the 10th percentile of density
vertices_to_remove = densities < np.quantile(densities, 0.1)
mesh.remove_vertices_by_mask(vertices_to_remove)
# ------------------------------------------------------------------
# `densities` tem um valor por vertice da malha ORIGINAL. Depois da
# poda a malha tem menos vertices, entao o vetor precisa ser podado
# junto - caso contrario cada vertice recebe a cor de OUTRO vertice e
# o mapa de confianca deixa de significar qualquer coisa.
densities = densities[~vertices_to_remove]
assert len(densities) == len(mesh.vertices), (
    f'densities ({len(densities)}) != vertices ({len(mesh.vertices)})')
# -----------------------------------------------------------------

print(f"✅ Reconstruction completed.")


🏗️ 3. Reconstructing Solid Surface (Poisson)...
   -> Raw mesh generated: 19866 vertices.
   -> Cutting ghosts and excesses...
✅ Reconstruction completed.


### ✨ **Digital Curation: Presentation of the Digital Twin**

The pinnacle of the processing pipeline is not limited to calculating the three-dimensional mesh, but presenting it with maximum physical and visual fidelity. The following code block extracts the consolidated matrix from the C++ engine (*Open3D*) and injects it into the interactive *Plotly* engine to forge the **Digital Twin** of the rescued anomaly.

In this stage, the technical rigor turns to the parameterization of **Physically Based Rendering (PBR)**:

**1. Normal Interpolation (*Smooth Shading*):**
By disabling flat shading (`flatshading = False`), the graphics engine stops rendering each mesh triangle individually. Instead, it interpolates the normals between the vertices, restoring the continuous and organic curvature typical of ceramic vessels or cast bronze artifacts.

**2. Material Photometry:**
The `lighting` dictionary is not purely aesthetic. The precise adjustment of the reflectance coefficients (diffuse and specular) associated with low roughness (`roughness = 0.1`) instructs the renderer to simulate the electromagnetic properties of Antique Gold or Bronze (`#B8860B`). The light interacts with the mathematical model exactly as it would reflect on the real metal.

**3. Museological Isolation:**
For the technical presentation, the total suppression of the Cartesian axes (`visible = False`) and the use of a black background (*dark room*) remove the "laboratory graph" bias. The result transcends a mere batch of stochastic data, delivering an immersive three-dimensional asset, ready to be integrated into engineering reports, archaeological repositories, or Virtual Reality (VR) exhibitions.

In [16]:
# ==========================================
# FINAL VISUALIZATION: THE RESCUED ARTIFACT
# ==========================================
verts = np.asarray(mesh.vertices)
tris = np.asarray(mesh.triangles)

fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x = verts[:, 0],
    y = verts[:, 1],
    z = verts[:, 2],
    i = tris[:, 0],
    j = tris[:, 1],
    k = tris[:, 2],
    color = '#B8860B',     # Dark Goldenrod (Antique Bronze/Gold Color)
    name = 'Reconstructed Amphora',
    opacity = 1.0,
    flatshading = False,   # Makes the light soft (Smooth Shading)
    lighting = dict(ambient = 0.4, diffuse = 0.6, roughness = 0.1, specular = 0.3)
))

fig.update_layout(
    title = "Final Result: Digital Reconstruction via AI",
    scene = dict(
        xaxis = dict(visible = False),
        yaxis = dict(visible = False),
        zaxis = dict(visible = False),
        aspectmode = 'data',
        bgcolor = 'black'
    ),
    paper_bgcolor = 'black',
    margin = dict(l = 0, r = 0, t = 40, b = 0)
)

fig.show()

### 📊 **Algorithmic Auditing: Reconstruction Confidence Mapping**

As proven in the previous steps, Poisson Reconstruction possesses an exceptional ability to fill gaps and interpolate continuous surfaces — successfully mitigating the 30% occlusion caused by the simulated sensor failure. However, from the rigorous perspective of **Scientific Metrology**, this extrapolation generates an analytical dilemma: when inspecting the final rendered mesh, it becomes visually impossible to distinguish which regions of the artifact are based on empirical data (real radar echoes) and which are mere "mathematical deductions" forged by the Differential Equation.

To solve this ethical and technical obstacle inherent to Computational Archaeology, we extract the energy density vector (`densities`), natively returned by the Open3D C++ engine. This vector acts as a rigorous **Confidence Score**.

The following code block converts the rendering engine into a visual auditing tool, generating a three-dimensional *Heatmap*:

**1. Scalar Normalization:** The raw densities calculated by the algorithm are normalized into a continuous mathematical spectrum from $0.0$ to $1.0$.

**2. Spectrum Projection (*Color Mapping*):** We resort to the perceptually uniform `Viridis` palette (via *Matplotlib*). The RGB color coordinates are extracted and surgically injected into each vertex of the 3D mesh through the `vertexcolor` parameter.

**3. Thermal Report Reading:** The chromatic scale reveals the structural "truth" behind the Digital Twin:
*   🟡 **Yellow / Light Green:** Represent zones of **extremely high confidence**, where the mesh is strongly anchored by real GPR points that survived the chaos filtering.
*   🟣 **Dark Blue / Purple:** Expose zones of **low confidence**, showing exactly the scars where the artifact was broken or occluded, requiring the mathematical intervention of the AI to close the topology.

In [17]:
# ============================================
# PHASE 5: SCIENTIFIC VISUALIZATION (HEATMAP)
# ============================================
print("\n📊 Generating Visual Confidence Report...")

verts = np.asarray(mesh.vertices)
tris = np.asarray(mesh.triangles)

# 1. Normalize densities to colors (0 to 1)
# densities comes from Poisson. The higher, the more original points support that face.
densities = np.asarray(densities)
assert len(densities) == len(verts), (
    'densities and vertices are misaligned - see the pruning cell above')
d_min, d_max = densities.min(), densities.max()
densities_norm = (densities - d_min) / (d_max - d_min)

# 2. Create Color Map (Matplotlib 'Jet' or 'Viridis')
# Viridis: Yellow (High Confidence) -> Purple (Low Confidence)
# plt.get_cmap is deprecated since matplotlib 3.7
cmap = matplotlib.colormaps['viridis']
colors = cmap(densities_norm)[:, :3]        # Get RGB, ignore Alpha

# 3. Plotly with Custom Vertex Colors
fig = go.Figure()

fig.add_trace(go.Mesh3d(
    x = verts[:, 0],
    y = verts[:, 1],
    z = verts[:, 2],
    i = tris[:, 0],
    j = tris[:, 1],
    k = tris[:, 2],
    vertexcolor = colors,
    name = 'Analytical Mesh',
    opacity = 1.0,
    lighting = dict(ambient = 0.5, diffuse = 0.5, roughness = 0.1, specular = 0.2)
))

# Adds a dummy color bar (visual workaround for Plotly to understand the scale)
fig.add_trace(go.Scatter3d(
    x = [None],
    y = [None],
    z = [None],
    mode = 'markers',
    marker = dict(
        colorscale = 'Viridis',
        cmin = d_min, cmax = d_max,
        showscale = True,
        colorbar = dict(title = 'Point Density (Confidence)')
    )
))

fig.update_layout(
    title = "Chronos Analytics: Reconstruction Confidence Map",
    scene = dict(
        xaxis = dict(visible = False),
        yaxis = dict(visible = False),
        zaxis = dict(visible = False),
        aspectmode = 'data',
        bgcolor = 'black'
    ),
    paper_bgcolor = 'black',
    font = dict(color = 'white')
)

fig.show()


📊 Generating Visual Confidence Report...


### 📏 **Final Metrology: Extraction of Physical Parameters and Virtual Report**

The processing cycle of the geometric engine culminates in the extraction of actionable metrics. With the Digital Twin stabilized and its integrity validated by the uncertainty map of the previous step, the structure ceases to be merely a visual representation and becomes a **mathematically measurable solid**.

To autonomously extract the dimensions of the artifact — without the need for human intervention or export to third-party CAD software —, the pipeline triggers the calculation of the **Axis-Aligned Bounding Box (AABB)** native to Open3D.

#### 🧮 **The Mathematics of Spatial Measurement**

The algorithm scans the vertex matrix of the continuous mesh (reconstructed by the Poisson equation), searching for the extreme values (minimum and maximum bounds) along the Euclidean space $\mathbb{R}^3$. The absolute difference between these vector coordinates returns the exact spatial proportions of the geometry:

*   **X Axis:** Maximum width of the artifact.
*   **Y Axis:** Depth (or thickness) of the structure.
*   **Z Axis:** Total height from base to top.

#### 📜 **The Virtual Archaeological Report**

The code consolidates these scalar quantities and issues an automated final report. This report is the definitive proof of the architectural viability of the system: we started the process in an extremely hostile simulated environment (with over 15,000 *backscatter* noise points and severe sensor failures), and the algorithm was able to sweep the chaos, purify the signal, calculate the closed mesh, audit the uncertainty, and measure the buried relic 100% autonomously.

In [18]:
# ==========================================
# FINAL PHASE: ARTIFACT METROLOGICAL REPORT
# ==========================================
print("\n📏 Extracting physical dimensions of the reconstructed artifact...")

# Get the mesh of the vessel that we generated with Poisson
bbox_vessel = mesh.get_axis_aligned_bounding_box()

# Extract the bounds
min_b = bbox_vessel.get_min_bound()
max_b = bbox_vessel.get_max_bound()

# Calculate the dimensions
width = max_b[0] - min_b[0]
depth = max_b[1] - min_b[1]
height = max_b[2] - min_b[2]

print(f"\n{'='*40}")
print(f"🏺 VIRTUAL ARCHAEOLOGICAL REPORT")
print(f"{'='*40}")
print(f"• Object Type: Solid of Revolution (Possible Amphora)")
print(f"• Reconstructed Vertices: {len(mesh.vertices)}")
print(f"• Maximum Dimensions (simulation units - the axes above are labelled")
print(f"  'm', so the report uses 'm' as well):")
print(f"    - X Axis (Width):  {width:.2f}")
print(f"    - Y Axis (Depth):  {depth:.2f}")
print(f"    - Z Axis (Height): {height:.2f}")

# --- Verify the topology before claiming anything about it -----------------
# "Watertight" is a property you check, not one you assert. This mesh is not
# watertight, and it cannot be: the density pruning that removes the Poisson
# bubble opens holes by construction. That is a reasonable engineering trade,
# but it means get_volume() is undefined - so falling back to a bounding box
# would report the envelope of whatever survived filtering, not the artifact.
# Report what you can compute; say plainly what you cannot.
_checks = {
    "watertight":        mesh.is_watertight(),
    "edge manifold":     mesh.is_edge_manifold(),
    "vertex manifold":   mesh.is_vertex_manifold(),
    "orientable":        mesh.is_orientable(),
    "self-intersecting": mesh.is_self_intersecting(),
}
print(f"\n• Mesh topology (verified, not assumed):")
for _k, _v in _checks.items():
    print(f"    - {_k:<20s} {'yes' if _v else 'no'}")

print(f"\n• Volumetry - four different quantities, reported separately:")
print(f"    - AABB envelope : {width * depth * height:10.2f}")
try:
    _hull, _ = mesh.compute_convex_hull()
    print(f"    - Convex hull   : {_hull.get_volume():10.2f}")
except Exception:
    pass
if _checks["watertight"]:
    print(f"    - True volume   : {mesh.get_volume():10.2f}")
else:
    print(f"    - True volume   :  undefined  (mesh is open after density pruning)")
print(f"{'='*40}")
print("✅ Chronos Project Part III completed successfully.")


📏 Extracting physical dimensions of the reconstructed artifact...

🏺 VIRTUAL ARCHAEOLOGICAL REPORT
• Object Type: Solid of Revolution (Possible Amphora)
• Reconstructed Vertices: 17879
• Maximum Dimensions (simulation units - the axes above are labelled
  'm', so report 'm' here too; v2.0 printed 'cm' for the same object):
    - X Axis (Width):  8.02
    - Y Axis (Depth):  8.04
    - Z Axis (Height): 6.11

• Mesh topology (verified, not assumed):
    - watertight           no
    - edge manifold        yes
    - vertex manifold      no
    - orientable           yes
    - self-intersecting    yes

• Volumetry - four different quantities, reported separately:
    - AABB envelope :     393.63
    - Convex hull   :     185.36
    - True volume   :  undefined  (mesh is open after density pruning)
✅ Chronos Project Part III completed successfully.


## 🏭 **Transition to Production: LIDAR Ingestion and Memory Optimization**

During all previous phases, we validated our mathematical engine using procedurally generated synthetic point clouds. However, for *Chronos* to act as a definitive field tool, it must be capable of ingesting raw scans from industrial sensors (LIDAR) and photogrammetric drones, whose data is typically stored in the binary standard `.las` or `.laz` (*LIDAR Data Exchange Format*).

---

### 💥 **The Computational Complexity Bottleneck (RAM)**

A modern topographic survey of an excavation site can easily contain **50 to 100 million spatial coordinates**. Injecting this matrix entirely into the statistical filtering functions (SOR) or the tensor calculation (surface normals) would result in unviable execution time and, inevitably, a RAM collapse (*Out of Memory Error*).

To mitigate this architectural obstacle, we developed a **Pre-Processing and Ingestion Module**. It acts on two tactical fronts:

1. **Binary Decoding:** Uses the native `laspy` library to break the compression of the raw file and extract the spatial vectors ($x, y, z$) directly into memory.
2. **Spatial Decimation (*Voxel Downsampling*):** Before delivering the data to the Artificial Intelligence phases, the *Open3D* engine applies a three-dimensional grid composed of virtual cubes (*Voxels*) over the entire cloud. The algorithm calculates the spatial average of all points falling within the same cube and condenses them into a single **mathematical centroid**.

### **Topological Preservation vs. Processing Load**

By calibrating the `downsample_voxel_size = 0.05` hyperparameter, we mathematically guarantee that the point cloud will have a maximum resolution of **5 centimeters**. This technique entirely preserves the macro-geometry of the ruin and the topology of the terrain, while **reducing the computational weight of the file by up to 90%**.

**The Architectural Leap:** This encapsulated module not only concludes our trials in this *Jupyter Notebook*, but also acts as the direct interface for code refactoring. It is the processing foundation that will allow the migration of this analytical ecosystem to a **Web Application in Production (`app.py`)**.

In [19]:
# pip install laspy[lazrs] open3d

In [20]:
# Real Data Ingestion Module
# Requires: pip install laspy[lazrs] open3d
# Only for study purposes, didn't even finished it here and went to app.py right away.
# But here's the basis for reading real LIDAR files, extracting coordinates, and optimizing for the reconstruction process.
import laspy

def read_las_file(file_path, downsample_voxel_size = 0.05):
    """
    Reads a LIDAR file (.las/.laz), extracts coordinates and applies
    spatial decimation (downsampling) to avoid RAM overflow.

    downsample_voxel_size: Size of the compression "cube" in meters (e.g., 0.05 = 5cm)
    """
    print(f"📂 Reading raw LIDAR file: {file_path}")

    try:
        las = laspy.read(file_path)
    except Exception as e:
        print(f"❌ Error reading the file: {e}")
        return None

    # 1. Raw coordinates extraction (with automatic scale adjustment from laspy)
    raw_points = np.vstack((las.x, las.y, las.z)).transpose()
    print(f"   -> Initial reading: {len(raw_points)} points captured.")

    # 2. Memory Optimization (Voxel Downsampling via Open3D)
    print(f"   -> Applying spatial compression (Voxel of {downsample_voxel_size}m)...")
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(raw_points)

    pcd_down = pcd.voxel_down_sample(voxel_size=downsample_voxel_size)
    optimized_points = np.asarray(pcd_down.points)

    reduction_rate = 100 - ((len(optimized_points) / len(raw_points)) * 100)
    print(f"✅ Optimized ingestion completed!")
    print(f"   -> Points retained for AI: {len(optimized_points)}")
    print(f"   -> Weight reduction: {reduction_rate:.1f}%")

    return optimized_points

# ========================================
# Example of how the pipeline would work:
# ========================================
# 1. Loads real data and already reduces the file size
# optimized_cloud = read_las_file("real_excavation_data.las", downsample_voxel_size = 0.1)

# 2. Passes directly to the phases we have already built
# pcd_clean = apply_statistical_filter(optimized_cloud)
# mesh_3d = reconstruct_poisson_surface(pcd_clean)

# 🏛️ **Conclusion: From the Mathematical Laboratory to Software Engineering in Production**

**Part III** of the *Chronos AI Project* consolidated the definitive transition from a two-dimensional proof of concept to an **Industry-Standard Geometric Engine**. Throughout this computational essay, we transcended the mere plotting of graphs: we taught the machine to interpret Euclidean space, surface topology, and the physics underlying raw geophysical data.

---

### 🏗️ **Synthesis of Architectural Development**

The evolution of our Computer Vision pipeline opened up the following frontiers of Data Science and Spatial Engineering:

* 📐 **1. Algebraic Foundation and Metrology:** We started with rigorous mathematical abstractions (`scipy.spatial` and `alphashape`). We proved the feasibility of extracting absolute metrics from buried anomalies, such as cubature calculation and mass estimation via *Convex Hull*.
* 🌪️ **2. Stochastic Resilience (*Stress Test*):** We abandoned the comfort of perfect data. We subjected the system to scenarios of extreme signal degradation, injecting sensor failures, volumetric noise (*backscatter*), and severe geological occlusions (low SNR).
* 🔬 **3. Micro-Excavation and Implicit Modeling:** We implemented morphological filters (SOR) for signal purification and solved Partial Differential Equations through **Poisson Surface Reconstruction**, culminating in a surgical topological pruning based on density.
* 📊 **4. Algorithmic Auditing and Digital Twins:** We developed Uncertainty Analysis tools (Confidence *Heatmaps*) and generated Autonomous Metrological Reports, attesting to the scientific integrity of the generated *Digital Twin*.

---

### 🚀 **The Leap to Production: The `app.py` Ecosystem**

Research in *Jupyter* fulfilled its vital role of theoretical foundation, prototyping, and algorithmic validation. However, for *Chronos* to fulfill its purpose of revolutionizing Archaeology and Field Engineering, mathematical complexity must be encapsulated in an accessible interface.

It is in this context that the project migrates to its next architectural stage: the evoluiton of the **Web Application in Production (`app.py`)**. Using the **Streamlit** *framework*, the entire pipeline validated in this laboratory will be converted into an interactive Graphical User Interface (GUI), integrating the old code and offering:

* 📂 **Drag-and-Drop Ingestion:** Professionals will be able to drag massive LIDAR or GPR files (`.las` / `.laz`) directly into the browser, without writing a single line of code.
* 🧠 **Autonomous Cloud Optimization:** The *Voxel Downsampling* module will protect the infrastructure against memory (RAM) collapses, intelligently processing spatial *Big Data*.
* 🎛️ **Dynamic Parameterization:** Complex variables (such as the aggressiveness of the statistical filter or the depth of the Poisson *Octree*) will be intuitively controlled by visual *sliders* in real time.
* 💾 **Universal Export:** One-click generation of `.obj` models, democratizing data access for 3D printing, CAD software, or Virtual Reality (VR) simulations.

---

### 🔮 **Vision of the Future: The Horizon of *Deep Learning* (Roadmap)**

The establishment of this volumetric reconstruction pipeline paves the way for the integration of **Purpose-Specific Artificial Intelligence** in future iterations of *Chronos*.

With the geometric foundation resolved, the tool's future will involve training **Three-Dimensional Convolutional Neural Networks (3D-CNNs)** and semantic segmentation architectures (such as *PointNet*). This will enable the system not only to reconstruct the artifact's mesh but to **autonomously classify it** (e.g., *"92% probability of being a 2nd-century Roman amphora"*).

---

## 🌐 **Appendix: Transdisciplinary Vision and Industrial Applications**

Although the *Chronos Project* was architected as a Computational Archaeology solution, the Computer Vision pipeline developed in this laboratory (LIDAR Ingestion $\rightarrow$ Stochastic Filtering $\rightarrow$ Poisson Reconstruction) forms the foundation of **Industry 4.0**.

The ability to transmute noisy point clouds into continuous and metrologically precise meshes has immediate scalability to three major sectors:

### **1. Digital Curation and Heritage Preservation**
Laser reconstruction has become the vanguard of historical preservation, enabling the creation of Virtual Museums (VR) and *Digital Twins*.
* **The Notre-Dame Paradigm:** After the tragic fire at Notre-Dame Cathedral in 2019, the millimeter precision of its architectural restoration was only possible thanks to LIDAR scans (point clouds) performed years earlier. The Poisson algorithm used in Chronos is exactly the class of mathematical model capable of converting these billions of points into the CAD model used by reconstruction engineers.

### **2. Civil Engineering, Mining, and Auditing**
In heavy industry, the volume extraction of anomalies (as we did in the tomb metrology stage) is monetized through **Cubature**.
* Mining giants use drone fleets to scan ore storage yards. By processing this point cloud with convex hull algorithms and implicit surfaces, the software calculates the exact volume of the pile. Our pipeline converts AI into an **Inventory Auditing** tool, where the mathematical volume calculation ($V$) directly determines the financial value of the yard.

### **3. Biomedical Engineering and Tomography**
The volumetric representation of occluded structures is the core of diagnostic imaging.
* A Computed Tomography (CT) or Magnetic Resonance Imaging (MRI) essentially generates a point cloud based on tissue density (bones, muscles, tumors). Sister algorithms to Poisson Reconstruction (such as *Marching Cubes*) are applied to this raw data to isolate the geometry of an organ, allowing surgeons to plan complex interventions in Virtual Reality or print custom titanium prosthetics on 3D printers.

The technology developed in this repository is not just a rescue of the past; it is the data infrastructure that builds the future.

---

### 📚 **References, Bibliography, and Technologies Used**

This spatial engineering laboratory was built on the shoulders of giants in computational geometry and computer vision. Below, we list the structural libraries and original theoretical foundations explored in this stage:

* **Geometric Engine and Computer Vision ([Open3D](http://www.open3d.org/docs/release/)):** Industry-standard framework (C++) used for vector field calculation (Normals), statistical filtering (SOR), *Voxel Downsampling*, and continuous meshes.
    * **Mathematical Foundation (Poisson):** Kazhdan, M., Bolitho, M., & Hoppe, H. (2006). [*Poisson Surface Reconstruction*](https://hhoppe.com/poissonrecon.pdf). Eurographics Symposium on Geometry Processing.
    * **Mathematical Foundation (BPA):** Bernardini, F., et al. (1999). [*The Ball-Pivoting Algorithm for Surface Reconstruction*](https://lidarwidgets.com/samples/bpa_tvcg.pdf). IEEE Transactions on Visualization and Computer Graphics.

* **Topology and Metrology ([SciPy Spatial Data Structures](https://docs.scipy.org/doc/scipy/reference/spatial.html)):** Official documentation of the spatial data structures used to calculate absolute volumes (Cubature), Delaunay Triangulation, and *Convex Hulls*.

* **Concave Hulls and Surface Tension ([Alpha Shapes](https://pypi.org/project/alphashape/)):** Python toolbox implemented for polygon generalization and extraction of strict concave meshes from point clouds.

* **LIDAR Data Ingestion ([Laspy](https://laspy.readthedocs.io/en/latest/)):** Structural library dedicated to reading, binary decoding, and processing laser scan files in standard geophysical industry formats (`.las` and `.laz`).

* **Graphic Rendering and PBR ([Plotly 3D Mesh](https://plotly.com/python/3d-mesh/)):** Visualization engine used for interactive projection of three-dimensional matrices, scalar uncertainty mapping (Heatmaps), and advanced photometric simulation (*Smooth Shading* and physically based lighting of materials).

---
